In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

UNIFIED_LABELS = ["Normal", "DR", "Glaucoma", "AMD", "Myopia"]
SPLIT_PATH = "/kaggle/input/datasets/bisusiddharth/edda-splits/splits/split_D123.csv"  # adjust to your actual path

df = pd.read_csv(SPLIT_PATH)
print(df.shape, df['split'].value_counts())

train_df = df[df['split']=='train'].reset_index(drop=True)
val_df   = df[df['split']=='val'].reset_index(drop=True)
test_df  = df[df['split']=='test'].reset_index(drop=True)

# Class weights (inverse frequency) - computed from train set
class_counts = train_df[UNIFIED_LABELS].sum()
total = len(train_df)
class_weights = torch.tensor([total/(len(UNIFIED_LABELS)*c) for c in class_counts], dtype=torch.float32).to(device)
print("Class weights:", dict(zip(UNIFIED_LABELS, class_weights.cpu().numpy())))

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

CUDA available: False
GPU not available


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())

CUDA available: True
GPU: Tesla T4
Number of GPUs: 2


In [13]:
import os
import pandas as pd

# Exact mounted dataset paths
D1_ROOT = "/kaggle/input/datasets/bisusiddharth/edda-d1-odir5k-clean/D1_clean"
D2_ROOT = "/kaggle/input/datasets/bisusiddharth/edda-d2-mured-clean/D2_clean"
D3_ROOT = "/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean"

# Load D123 split
SPLIT_PATH = "/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D123.csv"

df = pd.read_csv(SPLIT_PATH)

# Rebuild image paths for this Kaggle notebook
def actual_image_path(row):
    if row["source_dataset"] == "D1":
        return os.path.join(D1_ROOT, "images", row["filename"])
    elif row["source_dataset"] == "D2":
        return os.path.join(D2_ROOT, "images", row["filename"])
    elif row["source_dataset"] == "D3":
        return os.path.join(D3_ROOT, "images", row["filename"])
    else:
        raise ValueError(f"Unknown dataset: {row['source_dataset']}")

df["image_path"] = df.apply(actual_image_path, axis=1)

print("Shape:", df.shape)
print("Split counts:")
print(df["split"].value_counts())

print("Missing images:", (~df["image_path"].apply(os.path.exists)).sum())

Shape: (8401, 9)
Split counts:
split
train    5881
val      1260
test     1260
Name: count, dtype: int64
Missing images: 0


In [12]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    if filenames:
        print(dirname, "->", len(filenames), "files")

/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split -> 4 files
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits -> 8 files
/kaggle/input/datasets/bisusiddharth/edda-d1-odir5k-clean/D1_clean -> 10 files
/kaggle/input/datasets/bisusiddharth/edda-d1-odir5k-clean/D1_clean/images -> 6388 files
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean -> 8 files
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/images -> 3195 files
/kaggle/input/datasets/bisusiddharth/edda-d2-mured-clean/D2_clean -> 8 files
/kaggle/input/datasets/bisusiddharth/edda-d2-mured-clean/D2_clean/images -> 2204 files


In [14]:
UNIFIED_LABELS = ["Normal", "DR", "Glaucoma", "AMD", "Myopia"]

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

print("\nTraining class counts:")
print(train_df[UNIFIED_LABELS].sum())

Train: 5881
Val: 1260
Test: 1260

Training class counts:
Normal      2774
DR          1863
Glaucoma     671
AMD          400
Myopia       334
dtype: int64


In [17]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

UNIFIED_LABELS = ["Normal", "DR", "Glaucoma", "AMD", "Myopia"]
SPLIT_PATH = "/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D123.csv"  # adjust to your actual path

df = pd.read_csv(SPLIT_PATH)
print(df.shape, df['split'].value_counts())

train_df = df[df['split']=='train'].reset_index(drop=True)
val_df   = df[df['split']=='val'].reset_index(drop=True)
test_df  = df[df['split']=='test'].reset_index(drop=True)

# Class weights (inverse frequency) - computed from train set
class_counts = train_df[UNIFIED_LABELS].sum()
total = len(train_df)
class_weights = torch.tensor([total/(len(UNIFIED_LABELS)*c) for c in class_counts], dtype=torch.float32).to(device)
print("Class weights:", dict(zip(UNIFIED_LABELS, class_weights.cpu().numpy())))

Using: cuda
(8401, 9) split
train    5881
val      1260
test     1260
Name: count, dtype: int64
Class weights: {'Normal': np.float32(0.42400864), 'DR': np.float32(0.6313473), 'Glaucoma': np.float32(1.7529061), 'AMD': np.float32(2.9405), 'Myopia': np.float32(3.5215569)}


In [19]:
class EyeDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        img = self.transform(img)
        labels = torch.tensor(row[UNIFIED_LABELS].values.astype(np.float32))
        return img, labels

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_ds = EyeDataset(train_df, train_transform)
val_ds   = EyeDataset(val_df, eval_transform)
test_ds  = EyeDataset(test_df, eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [20]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Labels example:", labels[:3])

Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32, 5])
Labels example: tensor([[0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.]])


In [21]:
def build_model(arch_name, num_classes=5):
    if arch_name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        for p in model.features.parameters():
            p.requires_grad = False
        model.classifier[6] = nn.Linear(4096, num_classes)

    elif arch_name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V2")
        for p in model.parameters():
            p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif arch_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights="IMAGENET1K_V1")
        for p in model.features.parameters():
            p.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    return model.to(device)

In [22]:
def train_model(model, name, epochs=8):
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)

    best_val_f1 = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_f1 = evaluate(model, val_loader)
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Macro-F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f"{name}_best.pth")

    return model

def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            preds = (outputs > threshold).astype(int)
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    return f1_score(all_labels, all_preds, average='macro', zero_division=0)

def full_report(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            preds = (outputs > 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    results = {"Macro-F1": f1_score(all_labels, all_preds, average='macro', zero_division=0)}
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    for cls, f1 in zip(UNIFIED_LABELS, per_class_f1):
        results[f"{cls}_F1"] = f1
    return results

In [23]:
results_table = []

for arch in ["vgg16", "resnet50", "efficientnet_b3"]:
    print(f"\n{'='*50}\nTraining {arch}\n{'='*50}")
    model = build_model(arch)
    model = train_model(model, arch, epochs=8)  # 8 epochs, frozen backbone = fast

    model.load_state_dict(torch.load(f"{arch}_best.pth"))
    test_results = full_report(model, test_loader)
    test_results["Model"] = arch
    results_table.append(test_results)
    print(f"{arch} TEST results:", test_results)

results_df = pd.DataFrame(results_table)
print(results_df)
results_df.to_csv("/kaggle/working/baseline_results.csv", index=False)



Training vgg16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 196MB/s]  


[vgg16] Epoch 1/8 | Loss: 0.3979 | Val Macro-F1: 0.4283
[vgg16] Epoch 2/8 | Loss: 0.3340 | Val Macro-F1: 0.5147
[vgg16] Epoch 3/8 | Loss: 0.3070 | Val Macro-F1: 0.5950
[vgg16] Epoch 4/8 | Loss: 0.2954 | Val Macro-F1: 0.5571
[vgg16] Epoch 5/8 | Loss: 0.2775 | Val Macro-F1: 0.6124
[vgg16] Epoch 6/8 | Loss: 0.2636 | Val Macro-F1: 0.6293
[vgg16] Epoch 7/8 | Loss: 0.2569 | Val Macro-F1: 0.5916
[vgg16] Epoch 8/8 | Loss: 0.2461 | Val Macro-F1: 0.5808
vgg16 TEST results: {'Macro-F1': 0.633453981208112, 'Normal_F1': np.float64(0.7030411449016101), 'DR_F1': np.float64(0.5564924114671164), 'Glaucoma_F1': np.float64(0.5324675324675324), 'AMD_F1': np.float64(0.5419354838709678), 'Myopia_F1': np.float64(0.8333333333333334), 'Model': 'vgg16'}

Training resnet50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 211MB/s]


[resnet50] Epoch 1/8 | Loss: 0.5472 | Val Macro-F1: 0.0000
[resnet50] Epoch 2/8 | Loss: 0.4784 | Val Macro-F1: 0.0000
[resnet50] Epoch 3/8 | Loss: 0.4545 | Val Macro-F1: 0.0020
[resnet50] Epoch 4/8 | Loss: 0.4359 | Val Macro-F1: 0.0354
[resnet50] Epoch 5/8 | Loss: 0.4240 | Val Macro-F1: 0.0821
[resnet50] Epoch 6/8 | Loss: 0.4119 | Val Macro-F1: 0.1375
[resnet50] Epoch 7/8 | Loss: 0.4015 | Val Macro-F1: 0.1822
[resnet50] Epoch 8/8 | Loss: 0.3955 | Val Macro-F1: 0.2051
resnet50 TEST results: {'Macro-F1': 0.22726628794590148, 'Normal_F1': np.float64(0.1491628614916286), 'DR_F1': np.float64(0.24622030237580994), 'Glaucoma_F1': np.float64(0.1375), 'AMD_F1': np.float64(0.0), 'Myopia_F1': np.float64(0.603448275862069), 'Model': 'resnet50'}

Training efficientnet_b3
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 162MB/s] 


[efficientnet_b3] Epoch 1/8 | Loss: 0.5288 | Val Macro-F1: 0.0237
[efficientnet_b3] Epoch 2/8 | Loss: 0.4712 | Val Macro-F1: 0.0533
[efficientnet_b3] Epoch 3/8 | Loss: 0.4548 | Val Macro-F1: 0.1069
[efficientnet_b3] Epoch 4/8 | Loss: 0.4385 | Val Macro-F1: 0.1614
[efficientnet_b3] Epoch 5/8 | Loss: 0.4274 | Val Macro-F1: 0.2014
[efficientnet_b3] Epoch 6/8 | Loss: 0.4171 | Val Macro-F1: 0.2635
[efficientnet_b3] Epoch 7/8 | Loss: 0.4097 | Val Macro-F1: 0.2754
[efficientnet_b3] Epoch 8/8 | Loss: 0.4019 | Val Macro-F1: 0.2887
efficientnet_b3 TEST results: {'Macro-F1': 0.32542169215531, 'Normal_F1': np.float64(0.33418367346938777), 'DR_F1': np.float64(0.24839400428265523), 'Glaucoma_F1': np.float64(0.401673640167364), 'AMD_F1': np.float64(0.0), 'Myopia_F1': np.float64(0.6428571428571429), 'Model': 'efficientnet_b3'}
   Macro-F1  Normal_F1     DR_F1  Glaucoma_F1    AMD_F1  Myopia_F1  \
0  0.633454   0.703041  0.556492     0.532468  0.541935   0.833333   
1  0.227266   0.149163  0.246220     

In [24]:
class_counts = train_df[UNIFIED_LABELS].sum()

pos_weights = (len(train_df) - class_counts) / class_counts

pos_weights = torch.tensor(
    pos_weights.values,
    dtype=torch.float32
).to(device)

print("Pos weights:")
print(dict(zip(UNIFIED_LABELS, pos_weights.cpu().numpy())))

Pos weights:
{'Normal': np.float32(1.1200433), 'DR': np.float32(2.1567364), 'Glaucoma': np.float32(7.7645307), 'AMD': np.float32(13.7025), 'Myopia': np.float32(16.607784)}


In [25]:
def build_finetune_model(arch_name, num_classes=5):
    if arch_name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            num_classes
        )

    elif arch_name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V2")
        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif arch_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights="IMAGENET1K_V1")
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )

    return model.to(device)

In [26]:
def train_finetune_model(model, name, epochs=8):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    if name == "vgg16":
        backbone_params = model.features.parameters()
        head_params = model.classifier[6].parameters()

    elif name == "resnet50":
        backbone_params = [
            p for n, p in model.named_parameters()
            if not n.startswith("fc.")
        ]
        head_params = model.fc.parameters()

    elif name == "efficientnet_b3":
        backbone_params = model.features.parameters()
        head_params = model.classifier[1].parameters()

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": 1e-5},
            {"params": head_params, "lr": 1e-4}
        ],
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    best_val_f1 = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()

        val_f1 = evaluate(model, val_loader)
        avg_loss = total_loss / len(train_loader)

        print(
            f"[{name}] Epoch {epoch+1}/{epochs} | "
            f"Loss: {avg_loss:.4f} | "
            f"Val Macro-F1: {val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(
                model.state_dict(),
                f"{name}_finetune_best.pth"
            )

    return model

In [27]:
model = build_finetune_model("resnet50")

model = train_finetune_model(
    model,
    "resnet50",
    epochs=8
)

[resnet50] Epoch 1/8 | Loss: 1.0043 | Val Macro-F1: 0.4946
[resnet50] Epoch 2/8 | Loss: 0.7526 | Val Macro-F1: 0.5451
[resnet50] Epoch 3/8 | Loss: 0.6225 | Val Macro-F1: 0.6011
[resnet50] Epoch 4/8 | Loss: 0.5540 | Val Macro-F1: 0.6026
[resnet50] Epoch 5/8 | Loss: 0.5209 | Val Macro-F1: 0.6268
[resnet50] Epoch 6/8 | Loss: 0.4969 | Val Macro-F1: 0.6225
[resnet50] Epoch 7/8 | Loss: 0.4816 | Val Macro-F1: 0.6453
[resnet50] Epoch 8/8 | Loss: 0.4770 | Val Macro-F1: 0.6218


In [29]:
model.load_state_dict(torch.load("resnet50_finetune_best.pth"))

test_results = full_report(model, test_loader)

print("ResNet50 Fine-tuned TEST results:")
print(test_results)

ResNet50 Fine-tuned TEST results:
{'Macro-F1': 0.6453190809542597, 'Normal_F1': np.float64(0.774885145482389), 'DR_F1': np.float64(0.6487093153759821), 'Glaucoma_F1': np.float64(0.5804878048780487), 'AMD_F1': np.float64(0.4682274247491639), 'Myopia_F1': np.float64(0.7542857142857143)}


In [30]:
results_finetune = []

for arch in ["vgg16", "efficientnet_b3"]:

    print(f"\n{'='*50}")
    print(f"Fine-tuning {arch}")
    print(f"{'='*50}")

    model = build_finetune_model(arch)

    model = train_finetune_model(
        model,
        arch,
        epochs=8
    )

    model.load_state_dict(
        torch.load(f"{arch}_finetune_best.pth")
    )

    test_results = full_report(model, test_loader)

    results_finetune.append({
        "Model": arch,
        **test_results
    })

    print(f"{arch} TEST results:")
    print(test_results)


Fine-tuning vgg16
[vgg16] Epoch 1/8 | Loss: 0.8964 | Val Macro-F1: 0.5185
[vgg16] Epoch 2/8 | Loss: 0.6922 | Val Macro-F1: 0.5722
[vgg16] Epoch 3/8 | Loss: 0.5889 | Val Macro-F1: 0.5906
[vgg16] Epoch 4/8 | Loss: 0.5276 | Val Macro-F1: 0.6216
[vgg16] Epoch 5/8 | Loss: 0.4966 | Val Macro-F1: 0.6704
[vgg16] Epoch 6/8 | Loss: 0.4684 | Val Macro-F1: 0.6509
[vgg16] Epoch 7/8 | Loss: 0.4466 | Val Macro-F1: 0.6497
[vgg16] Epoch 8/8 | Loss: 0.4474 | Val Macro-F1: 0.6579
vgg16 TEST results:
{'Macro-F1': 0.6653964612730895, 'Normal_F1': np.float64(0.7756410256410257), 'DR_F1': np.float64(0.6653386454183267), 'Glaucoma_F1': np.float64(0.5150214592274678), 'AMD_F1': np.float64(0.5893719806763285), 'Myopia_F1': np.float64(0.7816091954022989)}

Fine-tuning efficientnet_b3
[efficientnet_b3] Epoch 1/8 | Loss: 1.0405 | Val Macro-F1: 0.4453
[efficientnet_b3] Epoch 2/8 | Loss: 0.9049 | Val Macro-F1: 0.4669
[efficientnet_b3] Epoch 3/8 | Loss: 0.8101 | Val Macro-F1: 0.4905
[efficientnet_b3] Epoch 4/8 | Los

In [33]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

UNIFIED_LABELS = ["Normal", "DR", "Glaucoma", "AMD", "Myopia"]
SPLIT_PATH = "/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D123.csv"

df = pd.read_csv(SPLIT_PATH)
train_df = df[df['split']=='train'].reset_index(drop=True)
val_df   = df[df['split']=='val'].reset_index(drop=True)
test_df  = df[df['split']=='test'].reset_index(drop=True)

class_counts = train_df[UNIFIED_LABELS].sum()
total = len(train_df)
class_weights = torch.tensor([total/(len(UNIFIED_LABELS)*c) for c in class_counts], dtype=torch.float32).to(device)
print("Class weights:", dict(zip(UNIFIED_LABELS, class_weights.cpu().numpy())))

Using: cuda
Class weights: {'Normal': np.float32(0.42400864), 'DR': np.float32(0.6313473), 'Glaucoma': np.float32(1.7529061), 'AMD': np.float32(2.9405), 'Myopia': np.float32(3.5215569)}


In [34]:
class EyeDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        img = self.transform(img)
        labels = torch.tensor(row[UNIFIED_LABELS].values.astype(np.float32))
        return img, labels

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_loader = DataLoader(EyeDataset(train_df, train_transform), batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(EyeDataset(val_df, eval_transform), batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(EyeDataset(test_df, eval_transform), batch_size=32, shuffle=False, num_workers=2)

In [35]:
def build_model(arch_name, num_classes=5):
    if arch_name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        for p in model.features.parameters():
            p.requires_grad = False
        for p in model.features[-6:].parameters():
            p.requires_grad = True
        model.classifier[6] = nn.Linear(4096, num_classes)

    elif arch_name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V2")
        for p in model.parameters():
            p.requires_grad = False
        for p in model.layer4.parameters():
            p.requires_grad = True
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif arch_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights="IMAGENET1K_V1")
        for p in model.features.parameters():
            p.requires_grad = False
        for p in model.features[-3:].parameters():
            p.requires_grad = True
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    return model.to(device)

def get_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            probs = torch.sigmoid(model(imgs)).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

def evaluate_f1(model, loader, thresholds=None):
    probs, labels = get_probs(model, loader)
    if thresholds is None:
        thresholds = [0.5]*len(UNIFIED_LABELS)
    preds = np.zeros_like(probs)
    for i, t in enumerate(thresholds):
        preds[:,i] = (probs[:,i] > t).astype(int)
    return f1_score(labels, preds, average='macro', zero_division=0)

def find_best_thresholds(model, loader):
    probs, labels = get_probs(model, loader)
    best_thresholds = []
    for i in range(len(UNIFIED_LABELS)):
        best_f1, best_t = 0, 0.5
        for t in np.arange(0.1, 0.9, 0.02):
            f1 = f1_score(labels[:,i], (probs[:,i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
    return best_thresholds

def full_report(model, loader, model_name, thresholds):
    probs, labels = get_probs(model, loader)
    preds = np.zeros_like(probs)
    for i, t in enumerate(thresholds):
        preds[:,i] = (probs[:,i] > t).astype(int)
    results = {"Model": model_name, "Macro-F1": f1_score(labels, preds, average='macro', zero_division=0)}
    per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
    for cls, f1 in zip(UNIFIED_LABELS, per_class_f1):
        results[f"{cls}_F1"] = f1
    return results

In [36]:
def train_model(model, name, epochs=15, patience=5):
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_val_f1 = 0
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_f1 = evaluate_f1(model, val_loader)
        scheduler.step(val_f1)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Macro-F1: {val_f1:.4f} | LR: {current_lr:.6f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{name}_best.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{name}] Early stopping at epoch {epoch+1}")
                break

    return model

In [37]:
results_table = []

for arch in ["vgg16", "resnet50", "efficientnet_b3"]:
    print(f"\n{'='*50}\nTraining {arch}\n{'='*50}")
    model = build_model(arch)
    model = train_model(model, arch, epochs=15, patience=5)
    model.load_state_dict(torch.load(f"{arch}_best.pth"))

    # Tune thresholds on VALIDATION set (not test - this is legitimate)
    best_thresholds = find_best_thresholds(model, val_loader)
    print(f"{arch} best thresholds:", dict(zip(UNIFIED_LABELS, best_thresholds)))

    # Evaluate on test set using those tuned thresholds
    test_results = full_report(model, test_loader, arch, best_thresholds)
    results_table.append(test_results)
    print(f"{arch} TEST results:", test_results)

results_df = pd.DataFrame(results_table)
print(results_df)
results_df.to_csv("/kaggle/working/baseline_results_improved.csv", index=False)


Training vgg16
[vgg16] Epoch 1/15 | Loss: 0.3667 | Val Macro-F1: 0.5754 | LR: 0.000100
[vgg16] Epoch 2/15 | Loss: 0.2857 | Val Macro-F1: 0.6229 | LR: 0.000100
[vgg16] Epoch 3/15 | Loss: 0.2651 | Val Macro-F1: 0.6549 | LR: 0.000100
[vgg16] Epoch 4/15 | Loss: 0.2422 | Val Macro-F1: 0.6386 | LR: 0.000100
[vgg16] Epoch 5/15 | Loss: 0.2245 | Val Macro-F1: 0.6587 | LR: 0.000100
[vgg16] Epoch 6/15 | Loss: 0.2152 | Val Macro-F1: 0.6918 | LR: 0.000100
[vgg16] Epoch 7/15 | Loss: 0.2040 | Val Macro-F1: 0.6753 | LR: 0.000100
[vgg16] Epoch 8/15 | Loss: 0.1913 | Val Macro-F1: 0.6882 | LR: 0.000100
[vgg16] Epoch 9/15 | Loss: 0.1834 | Val Macro-F1: 0.6353 | LR: 0.000050
[vgg16] Epoch 10/15 | Loss: 0.1615 | Val Macro-F1: 0.6933 | LR: 0.000050
[vgg16] Epoch 11/15 | Loss: 0.1461 | Val Macro-F1: 0.7157 | LR: 0.000050
[vgg16] Epoch 12/15 | Loss: 0.1370 | Val Macro-F1: 0.7105 | LR: 0.000050
[vgg16] Epoch 13/15 | Loss: 0.1331 | Val Macro-F1: 0.7126 | LR: 0.000050
[vgg16] Epoch 14/15 | Loss: 0.1271 | Val Mac

In [38]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

UNIFIED_LABELS = ["Normal", "DR", "Glaucoma", "AMD", "Myopia"]
SPLIT_PATH = "/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D123.csv"

df = pd.read_csv(SPLIT_PATH)
train_df = df[df['split']=='train'].reset_index(drop=True)
val_df   = df[df['split']=='val'].reset_index(drop=True)
test_df  = df[df['split']=='test'].reset_index(drop=True)

class_counts = train_df[UNIFIED_LABELS].sum()
total = len(train_df)
class_weights = torch.tensor([total/(len(UNIFIED_LABELS)*c) for c in class_counts], dtype=torch.float32).to(device)
print("Class weights:", dict(zip(UNIFIED_LABELS, class_weights.cpu().numpy())))

Using: cuda
Class weights: {'Normal': np.float32(0.42400864), 'DR': np.float32(0.6313473), 'Glaucoma': np.float32(1.7529061), 'AMD': np.float32(2.9405), 'Myopia': np.float32(3.5215569)}


In [39]:
IMG_SIZE = 300

class EyeDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        img = self.transform(img)
        labels = torch.tensor(row[UNIFIED_LABELS].values.astype(np.float32))
        return img, labels

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

BATCH_SIZE = 16  # reduced since 300x300 uses more memory than 224x224

train_loader = DataLoader(EyeDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(EyeDataset(val_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(EyeDataset(test_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [40]:
def build_model_full_finetune(arch_name, num_classes=5):
    if arch_name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        # ALL layers trainable now
        model.classifier[6] = nn.Linear(4096, num_classes)

    elif arch_name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V2")
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif arch_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights="IMAGENET1K_V1")
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    return model.to(device)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        focal = (1 - pt) ** self.gamma * bce
        if self.alpha is not None:
            focal = focal * self.alpha
        return focal.mean()

def get_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            probs = torch.sigmoid(model(imgs)).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

def evaluate_f1(model, loader, thresholds=None):
    probs, labels = get_probs(model, loader)
    if thresholds is None:
        thresholds = [0.5]*len(UNIFIED_LABELS)
    preds = np.zeros_like(probs)
    for i, t in enumerate(thresholds):
        preds[:,i] = (probs[:,i] > t).astype(int)
    return f1_score(labels, preds, average='macro', zero_division=0)

def find_best_thresholds(model, loader):
    probs, labels = get_probs(model, loader)
    best_thresholds = []
    for i in range(len(UNIFIED_LABELS)):
        best_f1, best_t = 0, 0.5
        for t in np.arange(0.05, 0.95, 0.02):
            f1 = f1_score(labels[:,i], (probs[:,i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
    return best_thresholds

def full_report(model, loader, model_name, thresholds):
    probs, labels = get_probs(model, loader)
    preds = np.zeros_like(probs)
    for i, t in enumerate(thresholds):
        preds[:,i] = (probs[:,i] > t).astype(int)
    results = {"Model": model_name, "Macro-F1": f1_score(labels, preds, average='macro', zero_division=0)}
    per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
    for cls, f1 in zip(UNIFIED_LABELS, per_class_f1):
        results[f"{cls}_F1"] = f1
    return results

In [41]:
def train_model_full(model, name, epochs=30, patience=7):
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)

    # Differential learning rates: backbone gets a smaller LR, new head gets a bigger LR
    if hasattr(model, 'classifier'):
        head_params = list(model.classifier.parameters())
        head_param_ids = set(id(p) for p in head_params)
        backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]
    else:  # resnet
        head_params = list(model.fc.parameters())
        head_param_ids = set(id(p) for p in head_params)
        backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]

    optimizer = torch.optim.Adam([
        {"params": backbone_params, "lr": 1e-5},   # small LR for pretrained weights
        {"params": head_params, "lr": 1e-4}         # bigger LR for new head
    ])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_val_f1 = 0
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_f1 = evaluate_f1(model, val_loader)
        scheduler.step(val_f1)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Macro-F1: {val_f1:.4f} | LR: {current_lr:.6f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{name}_full_best.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{name}] Early stopping at epoch {epoch+1} (best val F1: {best_val_f1:.4f})")
                break

    return model

In [42]:
results_table = []

for arch in ["vgg16", "resnet50", "efficientnet_b3"]:
    print(f"\n{'='*60}\nFull fine-tuning: {arch}\n{'='*60}")
    model = build_model_full_finetune(arch)
    model = train_model_full(model, arch, epochs=30, patience=7)
    model.load_state_dict(torch.load(f"{arch}_full_best.pth"))

    best_thresholds = find_best_thresholds(model, val_loader)
    print(f"{arch} best thresholds:", dict(zip(UNIFIED_LABELS, best_thresholds)))

    test_results = full_report(model, test_loader, arch, best_thresholds)
    results_table.append(test_results)
    print(f"{arch} TEST results:", test_results)

results_df = pd.DataFrame(results_table)
print(results_df)
results_df.to_csv("/kaggle/working/baseline_results_full_finetune.csv", index=False)


Full fine-tuning: vgg16
[vgg16] Epoch 1/30 | Loss: 0.1125 | Val Macro-F1: 0.5014 | LR: 0.000010
[vgg16] Epoch 2/30 | Loss: 0.0895 | Val Macro-F1: 0.6511 | LR: 0.000010
[vgg16] Epoch 3/30 | Loss: 0.0752 | Val Macro-F1: 0.6754 | LR: 0.000010
[vgg16] Epoch 4/30 | Loss: 0.0685 | Val Macro-F1: 0.6921 | LR: 0.000010
[vgg16] Epoch 5/30 | Loss: 0.0606 | Val Macro-F1: 0.6802 | LR: 0.000010
[vgg16] Epoch 6/30 | Loss: 0.0576 | Val Macro-F1: 0.7126 | LR: 0.000010
[vgg16] Epoch 7/30 | Loss: 0.0509 | Val Macro-F1: 0.7345 | LR: 0.000010
[vgg16] Epoch 8/30 | Loss: 0.0497 | Val Macro-F1: 0.7496 | LR: 0.000010
[vgg16] Epoch 9/30 | Loss: 0.0470 | Val Macro-F1: 0.7261 | LR: 0.000010
[vgg16] Epoch 10/30 | Loss: 0.0437 | Val Macro-F1: 0.7623 | LR: 0.000010
[vgg16] Epoch 11/30 | Loss: 0.0403 | Val Macro-F1: 0.7457 | LR: 0.000010
[vgg16] Epoch 12/30 | Loss: 0.0384 | Val Macro-F1: 0.7388 | LR: 0.000010
[vgg16] Epoch 13/30 | Loss: 0.0378 | Val Macro-F1: 0.7345 | LR: 0.000010
[vgg16] Epoch 14/30 | Loss: 0.0325 